# Irish Residential Mortgage Arrears - Data Pipeline (v2)

**Source:** Central Bank of Ireland, *Residential Mortgage Arrears and Repossession Statistics* (data.gov.ie, CC BY 4.0, quarterly, revised over time).

**Changes in this version (from mentor feedback)**

| # | Change | Where |
|---|---|---|
| 1 | **Historical split by target quarter**: train up to 2020Q4, dev 2021-22, **test 2023-24**, **2025-26 held back as `new_data`** for sprint 4 | CELL 15 |
| 2 | **Queryable storage**: BigQuery tables and views for fast, repeated EDA and feature work (GCS keeps raw files and Parquet exports) | CELLS 20-21 |
| 3 | Feature table is **versioned** (`features_v1`, `features_v2`, ...) so feature experiments never overwrite each other | CELLS 2, 20 |
| 4 | Data **vintage** column and raw-file SHA-256, because the publisher revises past figures | CELLS 6, 14, 22 |
| 5 | Bucket **object versioning** switched on | CELL 3 |

**Cell numbers** are written in the first comment of every code cell (`# CELL n`), and the config errors below refer to them.

## Section A - Setup (CELLS 0-6)

In [ ]:
# CELL 0 - Install dependencies (run once per runtime)
# pyarrow: Parquet + BigQuery loads from DataFrames | db-dtypes: BigQuery query results -> DataFrame
!pip install --quiet --upgrade google-cloud-storage google-cloud-bigquery pandas scikit-learn requests gcsfs pyarrow db-dtypes

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.5/79.5 kB 1.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.5/343.5 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.8/267.8 kB 20.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.8/10.8 MB 71.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 76.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 91.4/91.4 kB 10.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 17.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 221.7/221.7 kB 23.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires pandas==2.2.3, but you have pandas 3.0.6 which is incompatible.
google-colab 1.0.0 

In [ ]:
# CELL 1 - Imports, GCP settings and authentication
import os, re, json, hashlib, datetime, shutil
import numpy as np
import pandas as pd
import requests
from sklearn.model_selection import TimeSeriesSplit
from google.cloud import storage, bigquery

# --- GCP configuration ---
PROJECT_ID = "mortgages-forecasting-trends"
BUCKET_NAME = "mortgage_irish_forecast_trends"
LOCATION = "EU"                    # bucket and BigQuery dataset share this location
BQ_DATASET = "mortgage_arrears"    # BigQuery dataset that holds the queryable tables

# --- Authentication ---
# Option A (Colab / Vertex AI): interactive login
try:
    from google.colab import auth
    auth.authenticate_user()
except ImportError:
    # Option B (local): run `gcloud auth application-default login` in a terminal first
    # Option C (service account): os.environ["GOOGLE_APPLICATION_CREDENTIALS"] = "/path/key.json"
    print("Not running in Colab - using Application Default Credentials.")

print(f"Target project : {PROJECT_ID}")
print(f"Target bucket  : gs://{BUCKET_NAME}")
print(f"BigQuery dataset: {PROJECT_ID}.{BQ_DATASET}")

Target project : mortgages-forecasting-trends
Target bucket  : gs://mortgage_irish_forecast_trends
BigQuery dataset: mortgages-forecasting-trends.mortgage_arrears


In [ ]:
# CELL 2 - Pipeline configuration (the only cell you should need to edit)

# ---- Files and paths ----
LOCAL_RAW_PATH = "moa-open-data.csv"     # upload this file to Colab (Files panel) or let CELL 5 download it
RAW_SOURCE_URL = ("https://opendata.centralbank.ie/dataset/23075b1d-a112-4ae8-b1b2-f52eea3c2a6e/"
                  "resource/8013ade5-b6fa-4be7-8bb9-39bce48116c8/download/moa-open-data.csv")
LOCAL_DIR = "gcs_upload"                 # local staging folder that mirrors the bucket layout
EXPORT_FORMAT = "parquet"                # "parquet" (keeps dtypes, recommended) or "csv"

# ---- Versioning ----
FEATURE_VERSION = "v1"                   # bump to "v2" when you change the feature set
DATA_PULL_DATE = str(datetime.date.today())   # set the real download date if different (data is revised)

# ---- BigQuery object names (created in CELL 20) ----
TBL_TIDY = "moa_tidy_long"                       # cleaned long table, all categories
TBL_FEAT = f"features_{FEATURE_VERSION}"         # modelling feature table (has the split column)
TBL_COOP = f"coop_features_{FEATURE_VERSION}"    # co-operation features (2019+)

# ---- Modelling scope ----
ENTITY = "All"          # "All" = Banks + Non-Banks (national). Or "Banks" / "Non-Banks"
N_LAGS = (1, 2, 3, 4)   # lagged copies of the target used as features

# ---- Column names in the pivoted panel ("Category | Row") - verified in CELL 11 ----
ARREARS_TOTAL  = "Arrears | Arrears: Total mortgage accounts in arrears"
UP_TO_90       = "Arrears | Arrears: Total mortgage accounts in arrears - up to 90 days"
B91_180        = "Arrears | Arrears: Total mortgage accounts in arrears - 91 to 180 days"
OVER_180_ORIG  = "Arrears | Arrears: Total mortgage accounts in arrears - over 180 days"  # only 2009 - mid 2012
OVER_180_COMBINED = "Arrears | over180_combined"   # rebuilt in CELL 12 for the full period
STOCK_ROW      = None   # full "Category | Row" name of TOTAL mortgage accounts, if you find one in CELL 11
                        # None -> the target is the raw arrears count; set it to forecast the arrears RATE

# ---- Borrower co-operation category (only from 2019-09-30) ----
COOP_CATEGORY = "Borrower co-operation status"
LONG_BUCKETS = ["In Arrears 1 YR to 2 YR", "In Arrears 2 YR to 5 YR",
                "In Arrears 5 YR to 10 YR", "In Arrears over 10 YRS"]

# ---- Historical split, defined on the TARGET quarter (the quarter being predicted) ----
# A row's features come from quarter t and its label from quarter t+1, so splitting on the
# feature date would let a 2025 value leak into the 2024 test labels.
TRAIN_END      = pd.Period("2020Q4")     # train : targets up to and including this quarter
DEV_END        = pd.Period("2022Q4")     # dev   : 2021Q1 - 2022Q4
TEST_END       = pd.Period("2024Q4")     # test  : 2023Q1 - 2024Q4  (mentor's suggestion)
NEW_DATA_START = pd.Period("2025Q1")     # new_data: 2025Q1 onward, untouched until sprint 4

# ---- Cross-validation on TRAIN only (expanding window) ----
N_CV_SPLITS, CV_VAL_QUARTERS, CV_GAP = 5, 4, 1

assert TRAIN_END < DEV_END < TEST_END < NEW_DATA_START
assert TEST_END + 1 == NEW_DATA_START, "test and new_data windows must be contiguous"
print("Config OK | entity:", ENTITY, "| feature version:", FEATURE_VERSION)

Config OK | entity: All | feature version: v1


In [ ]:
# CELL 3 - Create the GCS bucket (if missing) and switch on object versioning
def create_gcs_bucket(bucket_name, project_id, location="EU"):
    """Create a GCS bucket if it does not already exist."""
    client = storage.Client(project=project_id)
    if client.lookup_bucket(bucket_name):
        print(f"Bucket 'gs://{bucket_name}' already exists.")
        return
    bucket = client.bucket(bucket_name)
    bucket.iam_configuration.uniform_bucket_level_access_enabled = True
    bucket = client.create_bucket(bucket, location=location)
    print(f"Created bucket: gs://{bucket.name}")

def enable_versioning(bucket_name, project_id):
    """Keep old object versions, so re-uploads never silently destroy earlier data."""
    bucket = storage.Client(project=project_id).get_bucket(bucket_name)
    if not bucket.versioning_enabled:
        bucket.versioning_enabled = True
        bucket.patch()
    print("Object versioning enabled:", bucket.versioning_enabled)

create_gcs_bucket(BUCKET_NAME, PROJECT_ID, LOCATION)
enable_versioning(BUCKET_NAME, PROJECT_ID)

Bucket 'gs://mortgage_irish_forecast_trends' already exists.
Object versioning enabled: True


In [ ]:
# CELL 4 - Helper: upload a local file to a GCS object path
def upload_to_gcs(bucket_name, source_file_name, destination_blob_name):
    """Upload a local file to gs://<bucket>/<destination_blob_name>."""
    client = storage.Client(project=PROJECT_ID)
    blob = client.bucket(bucket_name).blob(destination_blob_name)
    blob.upload_from_filename(source_file_name)
    print(f"Uploaded {source_file_name} -> gs://{bucket_name}/{destination_blob_name}")

In [ ]:
# CELL 5 - Get the raw CSV locally (manual upload preferred, download as a fallback)
if not os.path.exists(LOCAL_RAW_PATH):
    print(f"'{LOCAL_RAW_PATH}' not found - trying to download it from the Central Bank ...")
    try:
        r = requests.get(RAW_SOURCE_URL, timeout=60)
        r.raise_for_status()
        head = r.content[:300].decode("cp1252", errors="ignore")
        assert "Reporting Date" in head, "download does not look like the dataset (blocked or HTML page?)"
        with open(LOCAL_RAW_PATH, "wb") as f:
            f.write(r.content)
    except Exception as e:
        raise FileNotFoundError(
            f"Could not fetch the raw file ({e}). Download 'moa-open-data.csv' from data.gov.ie and "
            f"upload it to the Colab Files panel, then re-run this cell.")

print(f"Raw file ready: {LOCAL_RAW_PATH} ({os.path.getsize(LOCAL_RAW_PATH):,} bytes)")

Raw file ready: moa-open-data.csv (2,367,862 bytes)


In [ ]:
# CELL 6 - Fingerprint the raw file and archive it untouched in gs://<bucket>/raw/
def sha256(path):
    """SHA-256 of a file, so processed data can be tied to an exact raw download."""
    h = hashlib.sha256()
    with open(path, "rb") as f:
        for chunk in iter(lambda: f.read(1 << 20), b""):
            h.update(chunk)
    return h.hexdigest()

RAW_SHA256 = sha256(LOCAL_RAW_PATH)
print("Raw SHA-256:", RAW_SHA256)
upload_to_gcs(BUCKET_NAME, LOCAL_RAW_PATH, "raw/moa-open-data.csv")

Raw SHA-256: 67b8ac6b677928f8a73d90a5a3bfee77c7e0a4bd707add7833bd7533ce1540ee
Uploaded moa-open-data.csv -> gs://mortgage_irish_forecast_trends/raw/moa-open-data.csv


## Section B - Load, clean and validate (CELLS 7-9)

In [ ]:
# CELL 7 - Load the raw CSV and check for non-numeric values (diagnostic only, changes nothing)
# cp1252 because the euro sign in "Arrears EUR (000)" is a Windows-1252 character
df = pd.read_csv(LOCAL_RAW_PATH, encoding="cp1252")
df.columns = df.columns.str.strip()          # header has padded names like " Number of Accounts "
print("Shape:", df.shape)

for c in ["Observation", "Number of Accounts"]:
    print(f"\n{c}: dtype = {df[c].dtype}")
    txt = df[c].dropna().astype(str).str.replace(",", "", regex=False).str.strip()
    bad = txt[pd.to_numeric(txt, errors="coerce").isna()]
    print("  non-numeric values:", bad.unique()[:20], "| count:", len(bad))

Shape: (19199, 10)

Observation: dtype = str
  non-numeric values: <ArrowStringArray>
[]
Length: 0, dtype: str | count: 0

Number of Accounts: dtype = str
  non-numeric values: <ArrowStringArray>
[]
Length: 0, dtype: str | count: 0


In [ ]:
# CELL 8 - Clean and tidy into one numeric "value" column with a "unit"
df["Reporting Date"] = pd.to_datetime(df["Reporting Date"], dayfirst=True)   # source is dd/mm/yyyy
for c in ["Entity Group", "Format", "Category", "Row", "Observation Type"]:
    df[c] = df[c].astype(str).str.strip()

def to_num(s):
    """Text -> float: strip thousands separators, anything unparseable becomes NaN."""
    out = pd.to_numeric(
        s.astype("string").str.replace(",", "", regex=False).str.strip(),
        errors="coerce",
    )
    return pd.Series(out.to_numpy(dtype="float64", na_value=np.nan), index=s.index)

# money sits in "Observation", account counts in "Number of Accounts" -> merge into one column
df["value"] = to_num(df["Observation"]).fillna(to_num(df["Number of Accounts"]))

def make_unit(s):
    """Collapse the three standard types; keep co-operation / restructure types as separate measures."""
    if s.startswith("Balance"): return "Balance"
    if s.startswith("Arrears"): return "Arrears"
    if s == "Number":           return "Number"
    return s

df["unit"] = df["Observation Type"].map(make_unit)
df.loc[df["unit"].isin(["Balance", "Arrears"]), "value"] *= 1000   # EUR (000) -> EUR

print("value dtype:", df["value"].dtype)
print(df["unit"].value_counts())

value dtype: float64
unit
Number                                           6855
Balance                                          5711
Arrears                                          5121
Not Co-operating, Legal Proceed. Initiate         378
Not Co-operating, Legal Proceed. Not Initiate     378
Number, In Arrears, Restruc.                      378
Number, Not Restruc., Co-operating                378
Name: count, dtype: int64


In [ ]:
# CELL 9 - Data-quality gate: stop here if the tidy table is not safe to pivot
keys = ["Reporting Date", "Entity Group", "Format", "Category", "Row Position", "Row", "unit"]

n_dup = int(df.duplicated(keys).sum())
n_missing = int(df["value"].isna().sum())
print("Duplicate keys        :", n_dup)
print("Rows with no value    :", n_missing)
print("Date range            :", df["Reporting Date"].min().date(), "->", df["Reporting Date"].max().date())
print("Entity groups         :", sorted(df["Entity Group"].unique()))
print("Formats               :", sorted(df["Format"].unique()))

if n_dup:   # show the offenders before failing
    display(df[df.duplicated(keys, keep=False)].sort_values(keys).head(20))

assert df["value"].dtype == "float64"
assert n_dup == 0, "Duplicate keys - inspect the rows above (usually an Observation Type collapsed by make_unit)"

Duplicate keys        : 0
Rows with no value    : 0
Date range            : 2009-09-30 -> 2026-03-31
Entity groups         : ['Banks', 'Non-Banks']
Formats               : ['BTL', 'PDH']


## Section C - Panel and features (CELLS 10-14)

In [ ]:
# CELL 10 - Pivot account counts into a panel: one row per (quarter, entity, format)
IDX = ["Reporting Date", "Entity Group", "Format"]

def add_all(p):
    """Add an 'All' entity = Banks + Non-Banks, so loan-book sales between them cancel out."""
    nat = p.groupby(level=["Reporting Date", "Format"]).sum(min_count=1)
    nat["Entity Group"] = "All"
    nat = nat.set_index("Entity Group", append=True).reorder_levels(IDX)
    return pd.concat([p, nat]).sort_index()

num = df[df["unit"] == "Number"].copy()
num["measure"] = num["Category"] + " | " + num["Row"]     # prefix with category so names cannot collide

panel = num.pivot_table(index=IDX, columns="measure", values="value", aggfunc="first").sort_index()
panel = add_all(panel)

print("Panel shape:", panel.shape)
print("\nRows per (entity, format):")
print(panel.groupby(level=["Entity Group", "Format"]).size())

Panel shape: (369, 39)

Rows per (entity, format):
Entity Group  Format
All           BTL       56
              PDH       67
Banks         BTL       56
              PDH       67
Non-Banks     BTL       56
              PDH       67
dtype: int64


In [ ]:
# CELL 11 - Verify the column names in CELL 2 exist, and see when each arrears row is reported
print("Panel columns:")
for c in panel.columns:
    print("  ", c)

print()
for name in ["ARREARS_TOTAL", "UP_TO_90", "B91_180", "OVER_180_ORIG"]:
    val = globals()[name]
    print(f"{name:14s}: {'OK' if val in panel.columns else 'NOT FOUND - fix in CELL 2'}")
if STOCK_ROW is not None:
    print(f"{'STOCK_ROW':14s}: {'OK' if STOCK_ROW in panel.columns else 'NOT FOUND - fix in CELL 2'}")

# coverage of each arrears row (the buckets were redefined in 2012, so some rows stop or start)
p = panel.xs(ENTITY, level="Entity Group")
rows = []
for c in [c for c in panel.columns if c.startswith("Arrears |")]:
    s = p[c].dropna()
    d = s.index.get_level_values("Reporting Date")
    rows.append((c.replace("Arrears | ", ""), len(s), d.min().date(), d.max().date()))
pd.set_option("display.max_colwidth", 100)
display(pd.DataFrame(rows, columns=["row", "n_rows", "first", "last"]).sort_values("first"))

for name in ["ARREARS_TOTAL", "UP_TO_90", "B91_180"]:
    assert globals()[name] in panel.columns, f"{name} not found - fix CELL 2"

Panel columns:
   Arrears | Arrears: Total mortgage accounts in arrears
   Arrears | Arrears: Total mortgage accounts in arrears - 181 to 365 days
   Arrears | Arrears: Total mortgage accounts in arrears - 2 to 5 yrs
   Arrears | Arrears: Total mortgage accounts in arrears - 365 to 730 days (1 to 2 YRS)
   Arrears | Arrears: Total mortgage accounts in arrears - 5 to 10 yrs
   Arrears | Arrears: Total mortgage accounts in arrears - 91 to 180 days
   Arrears | Arrears: Total mortgage accounts in arrears - over 10 yrs
   Arrears | Arrears: Total mortgage accounts in arrears - over 180 days
   Arrears | Arrears: Total mortgage accounts in arrears - over 720 days (2 YRS)
   Arrears | Arrears: Total mortgage accounts in arrears - up to 90 days
   Duration in Legal Proceedings | Over 1 year and up to 2 years since first court hearing
   Duration in Legal Proceedings | Over 2 years and up to 5 years since first court hearing
   Duration in Legal Proceedings | Over 5 years since first court hea

,row,n_rows,first,last
0,Arrears: Total mortgage accounts in arrears,123,2009-09-30,2026-03-31
5,Arrears: Total mortgage accounts in arrears - 91 to 180 days,123,2009-09-30,2026-03-31
7,Arrears: Total mortgage accounts in arrears - over 180 days,13,2009-09-30,2012-06-29
9,Arrears: Total mortgage accounts in arrears - up to 90 days,123,2009-09-30,2026-03-31
1,Arrears: Total mortgage accounts in arrears - 181 to 365 days,110,2012-09-28,2026-03-31
3,Arrears: Total mortgage accounts in arrears - 365 to 730 days (1 to 2 YRS),110,2012-09-28,2026-03-31
8,Arrears: Total mortgage accounts in arrears - over 720 days (2 YRS),56,2012-09-28,2019-06-28
2,Arrears: Total mortgage accounts in arrears - 2 to 5 yrs,54,2019-09-30,2026-03-31
4,Arrears: Total mortgage accounts in arrears - 5 to 10 yrs,54,2019-09-30,2026-03-31
6,Arrears: Total mortgage accounts in arrears - over 10 yrs,54,2019-09-30,2026-03-31


In [ ]:
# CELL 12 - Rebuild "over 180 days" for the full period
# The reported over-180 row only exists for 2009 - mid 2012. Because the buckets add up to the total:
#   over_180 = total in arrears - up to 90 days - 91 to 180 days
OVER_180_METHOD = "residual: total - up_to_90 - 91_to_180"
OVER_180_RESID = panel[ARREARS_TOTAL] - panel[UP_TO_90] - panel[B91_180]

# validate: where the reported row exists, the residual must match it
cmp = pd.concat([OVER_180_RESID.rename("resid"), panel[OVER_180_ORIG].rename("orig")], axis=1).dropna()
OVER_180_OVERLAP_MAX_DIFF = float((cmp["resid"] - cmp["orig"]).abs().max()) if len(cmp) else None
print("Overlap rows:", len(cmp), "| max abs diff:", OVER_180_OVERLAP_MAX_DIFF)
if OVER_180_OVERLAP_MAX_DIFF:   # not 0 -> the identity does not hold, do not trust the rebuild
    print("WARNING: residual does not match the reported row - inspect before continuing")

panel[OVER_180_COMBINED] = OVER_180_RESID     # re-runnable: the original row is never overwritten
print("Rows with over-180 now:", panel[OVER_180_COMBINED].notna().sum(), "of", len(panel))

Overlap rows: 39 | max abs diff: 0.0
Rows with over-180 now: 368 of 369


In [ ]:
# CELL 13 - Co-operation / restructure features (data starts 2019-09-30, so descriptive use only)
coop = df[(df["Category"] == COOP_CATEGORY) & (~df["unit"].isin(["Balance", "Arrears"]))].copy()

status = coop.pivot_table(index=IDX, columns="unit", values="value", aggfunc="sum")
long_arrears = (coop[coop["Row"].isin(LONG_BUCKETS)]
                .groupby(IDX)["value"].sum().rename("long_arrears_accts"))

coop_counts = status.join(long_arrears)
coop_counts = add_all(coop_counts)            # combine Banks + Non-Banks on COUNTS first, then form ratios

coop_feat = pd.DataFrame(index=coop_counts.index)
coop_feat["coop_total"] = coop_counts[status.columns].sum(axis=1)

def share(col):
    """Share of accounts in a given status (NaN if that status is absent from the data)."""
    return coop_counts[col] / coop_feat["coop_total"] if col in coop_counts else np.nan

coop_feat["long_arrears_share"]    = coop_counts["long_arrears_accts"] / coop_feat["coop_total"]
coop_feat["restructured_share"]    = share("Number, In Arrears, Restruc.")
coop_feat["not_coop_legal_init"]   = share("Not Co-operating, Legal Proceed. Initiate")
coop_feat["not_coop_legal_noinit"] = share("Not Co-operating, Legal Proceed. Not Initiate")

d = coop_feat.index.get_level_values("Reporting Date")
print("Co-op data:", d.min().date(), "->", d.max().date())

# reconciliation: 0 means the statuses partition the arrears accounts, so the shares are valid
diff = (coop_feat["coop_total"] - panel[ARREARS_TOTAL]).dropna().abs()
print(f"Max abs diff, coop total vs arrears total: {diff.max():,.0f}  (0 = statuses partition arrears)")
display(coop_feat.xs(ENTITY, level="Entity Group").tail())

Co-op data: 2019-09-30 -> 2026-03-31
Max abs diff, coop total vs arrears total: 926,295  (0 = statuses partition arrears)


,,coop_total,long_arrears_share,restructured_share,not_coop_legal_init,not_coop_legal_noinit
Reporting Date,Format,,,,,
2025-03-31,PDH,40332.0,0.465313,0.241000,0.107458,0.162080
2025-06-30,PDH,37750.0,0.477987,0.241616,0.147629,0.170331
2025-09-30,PDH,36106.0,0.486540,0.239074,0.119897,0.162383
2025-12-01,PDH,35232.0,0.458532,0.242393,0.127810,0.156761
2026-03-31,PDH,34996.0,0.451680,0.237084,0.132444,0.153732


In [ ]:
# CELL 14 - Main feature table (arrears-only features: ~60 quarters of history)
GROUP_LEVELS = ["Entity Group", "Format"]

# guard: shift(-1) and lags are positional, so every series must have NO missing quarters
for grp_key, g in panel[ARREARS_TOTAL].dropna().groupby(level=GROUP_LEVELS):
    q = g.index.get_level_values("Reporting Date").to_period("Q")
    assert (np.diff(q.asi8) == 1).all(), f"missing quarter(s) in series {grp_key}"

feat = pd.DataFrame(index=panel.index)
feat["arrears_total"]  = panel[ARREARS_TOTAL]
feat["over180_share"]  = panel[OVER_180_COMBINED] / panel[ARREARS_TOTAL]
feat["b91_180_growth"] = panel.groupby(level=GROUP_LEVELS)[B91_180].pct_change(fill_method=None)

if STOCK_ROW is not None:
    feat["arrears_rate"] = panel[ARREARS_TOTAL] / panel[STOCK_ROW]
    TARGET = "arrears_rate"
else:
    TARGET = "arrears_total"
print("Forecast target:", TARGET)

grp = feat.groupby(level=GROUP_LEVELS)
feat["target_next_q"] = grp[TARGET].shift(-1)                 # label = next quarter's value
for lag in N_LAGS:
    feat[f"{TARGET}_lag{lag}"] = grp[TARGET].shift(lag)       # past information only

feat = feat.replace([np.inf, -np.inf], np.nan)

# require the FEATURES, not the label: the newest quarter has no label yet but is the row you
# actually predict from, so it must survive (it goes to the new_data split)
feature_cols = ["arrears_total", "over180_share", "b91_180_growth"] + [f"{TARGET}_lag{l}" for l in N_LAGS]
feat = feat.dropna(subset=feature_cols).reset_index()
feat = feat[feat["Entity Group"] == ENTITY].sort_values(["Reporting Date", "Format"]).reset_index(drop=True)

# vintage / version columns: past figures get revised, so record which pull and feature set made each row
feat["data_vintage"] = DATA_PULL_DATE
feat["feature_version"] = FEATURE_VERSION

print("Feature table:", feat.shape)
print("Quarters:", feat["Reporting Date"].nunique(), "|",
      feat["Reporting Date"].min().date(), "->", feat["Reporting Date"].max().date())
print("Rows without a label (latest quarter):", int(feat["target_next_q"].isna().sum()))
display(feat.tail())

Forecast target: arrears_total
Feature table: (115, 13)
Quarters: 63 | 2010-09-30 -> 2026-03-31
Rows without a label (latest quarter): 2


,Reporting Date,Entity Group,Format,arrears_total,over180_share,b91_180_growth,target_next_q,arrears_total_lag1,arrears_total_lag2,arrears_total_lag3,arrears_total_lag4,data_vintage,feature_version
110,2025-09-30,All,PDH,36106.0,0.575140,-0.035223,35232.0,37750.0,40332.0,42384.0,43537.0,2026-09-30,v1
111,2025-12-01,All,BTL,5786.0,0.745074,-0.049844,50508.0,6301.0,6150.0,6648.0,7190.0,2026-09-30,v1
112,2025-12-01,All,PDH,35232.0,0.539254,-0.059409,948411.0,36106.0,37750.0,40332.0,42384.0,2026-09-30,v1
113,2026-03-31,All,BTL,50508.0,-338.998495,56298.996721,NaN,5786.0,6301.0,6150.0,6648.0,2026-09-30,v1
114,2026-03-31,All,PDH,948411.0,-0.196113,367.485533,NaN,35232.0,36106.0,37750.0,40332.0,2026-09-30,v1


## Section D - Historical split and cross-validation (CELLS 15-16)

In [ ]:
# CELL 15 - Historical split by TARGET quarter (mentor: test = 2023-24, 2025-26 kept as new data)
rep_q = feat["Reporting Date"].dt.to_period("Q")
tq = rep_q + 1                                    # the quarter each row is predicting
feat["target_quarter"] = tq.astype(str)           # string form, safe for Parquet and BigQuery
labeled = feat["target_next_q"].notna()

feat["split"] = np.select(
    [labeled & (tq <= TRAIN_END),
     labeled & (tq > TRAIN_END) & (tq <= DEV_END),
     labeled & (tq > DEV_END) & (tq <= TEST_END),
     tq >= NEW_DATA_START],
    ["train", "dev", "test", "new_data"],
    default="unassigned")

summary = feat.groupby("split").agg(rows=("Reporting Date", "size"),
                                    feature_quarters=("Reporting Date", "nunique"),
                                    first_feature=("Reporting Date", "min"),
                                    last_feature=("Reporting Date", "max"),
                                    first_target=("target_quarter", "min"),
                                    last_target=("target_quarter", "max"))
print(summary.loc[[s for s in ["train", "dev", "test", "new_data", "unassigned"] if s in summary.index]])

# ---- leakage and completeness checks ----
assert not (feat["split"] == "unassigned").any(), "rows fall outside the split rules"
order = ["train", "dev", "test", "new_data"]
for s in order:
    assert (feat["split"] == s).any(), f"split '{s}' is empty"
for a, b in zip(order, order[1:]):
    a_max = pd.PeriodIndex(feat.loc[feat.split == a, "target_quarter"], freq="Q").max()
    b_min = pd.PeriodIndex(feat.loc[feat.split == b, "target_quarter"], freq="Q").min()
    assert a_max < b_min, f"target quarters of '{a}' and '{b}' overlap"
assert feat.loc[feat["target_next_q"].isna(), "split"].eq("new_data").all(), "unlabeled row outside new_data"
assert feat.loc[feat.split.isin(["train", "dev", "test"]), "target_next_q"].notna().all()

train_df = feat[feat.split == "train"].copy()
dev_df   = feat[feat.split == "dev"].copy()
test_df  = feat[feat.split == "test"].copy()
new_df   = feat[feat.split == "new_data"].copy()

print("\nTarget mean by split (drift is expected - arrears fell sharply after 2013):")
print(feat[feat.split != "new_data"].groupby("split")["target_next_q"].mean().reindex(["train", "dev", "test"]))
print("\nNOTE: co-operation features start 2019Q3, so only a handful of train quarters have them.")
print("      Use them descriptively (goal 3), not as inputs to a trained model on this split.")

          rows  feature_quarters first_feature last_feature first_target  \
split                                                                      
train       71                41    2010-09-30   2020-09-30       2010Q4   
dev         16                 8    2020-12-31   2022-09-30       2021Q1   
test        16                 8    2022-12-30   2024-09-30       2023Q1   
new_data    12                 6    2024-12-31   2026-03-31       2025Q1   

         last_target  
split                 
train         2020Q4  
dev           2022Q4  
test          2024Q4  
new_data      2026Q2  

Target mean by split (drift is expected - arrears fell sharply after 2013):
split
train    65443.746479
dev      30365.562500
test     27694.687500
Name: target_next_q, dtype: float64

NOTE: co-operation features start 2019Q3, so only a handful of train quarters have them.
      Use them descriptively (goal 3), not as inputs to a trained model on this split.


In [ ]:
# CELL 16 - Expanding-window cross-validation on TRAIN only (whole quarters stay together)
# gap=1 is a conservative buffer between the end of each training window and its validation block
train_quarters = np.sort(train_df["Reporting Date"].unique())
assert len(train_quarters) > CV_GAP + CV_VAL_QUARTERS * N_CV_SPLITS + 8, \
    "Too few train quarters - lower N_CV_SPLITS or CV_VAL_QUARTERS in CELL 2"

tscv = TimeSeriesSplit(n_splits=N_CV_SPLITS, test_size=CV_VAL_QUARTERS, gap=CV_GAP)
cv_folds = {}
for k, (tr_idx, va_idx) in enumerate(tscv.split(train_quarters)):
    tr_q, va_q = train_quarters[tr_idx], train_quarters[va_idx]
    cv_folds[k] = (train_df[train_df["Reporting Date"].isin(tr_q)].copy(),
                   train_df[train_df["Reporting Date"].isin(va_q)].copy())
    print(f"fold {k}: train {len(tr_q):2d}q ends {pd.Timestamp(tr_q[-1]).date()} | "
          f"val {pd.Timestamp(va_q[0]).date()} -> {pd.Timestamp(va_q[-1]).date()}")

fold 0: train 20q ends 2015-06-30 | val 2015-12-31 -> 2016-09-30
fold 1: train 24q ends 2016-06-30 | val 2016-12-30 -> 2017-09-29
fold 2: train 28q ends 2017-06-30 | val 2017-12-29 -> 2018-09-28
fold 3: train 32q ends 2018-06-29 | val 2018-12-31 -> 2019-09-30
fold 4: train 36q ends 2019-06-28 | val 2019-12-31 -> 2020-09-30


## Section E - Export to GCS (CELLS 17-19)

In [ ]:
# CELL 17 - Write the splits and CV folds locally in the bucket layout (columns renamed to snake_case)
COLUMN_RENAMES = {"format": "loan_format", "row": "row_label"}   # avoid SQL keyword-like column names

def snake(c):
    """'Reporting Date' -> 'reporting_date' (same names in Parquet files and BigQuery)."""
    c = c.strip().lower()
    for a, b in [(" | ", "_"), (" ", "_"), ("-", "_"), (":", ""), ("/", "_")]:
        c = c.replace(a, b)
    return COLUMN_RENAMES.get(re.sub(r"_+", "_", c), re.sub(r"_+", "_", c))

def to_snake_frame(d):
    out = d.copy()
    out.columns = [snake(c) for c in out.columns]
    return out

def write_frame(d, rel_path_no_ext):
    """Write a DataFrame under LOCAL_DIR as parquet or csv; returns the path."""
    path = f"{LOCAL_DIR}/{rel_path_no_ext}.{EXPORT_FORMAT}"
    os.makedirs(os.path.dirname(path), exist_ok=True)
    d.to_parquet(path, index=False) if EXPORT_FORMAT == "parquet" else d.to_csv(path, index=False)
    return path

def read_frame(path):
    if path.endswith(".parquet"):
        return pd.read_parquet(path)
    return pd.read_csv(path, parse_dates=["reporting_date"])

shutil.rmtree(LOCAL_DIR, ignore_errors=True)      # start clean so stale files are never uploaded
os.makedirs(f"{LOCAL_DIR}/sharded/cv_folds")

export_frames = {name: to_snake_frame(d) for name, d in
                 [("train", train_df), ("dev", dev_df), ("test", test_df), ("new_data", new_df)]}
for name, d in export_frames.items():
    write_frame(d, f"sharded/{name}")

for k, (tr, va) in cv_folds.items():
    for role, part in [("train", tr), ("val", va)]:
        out = to_snake_frame(part.drop(columns=["split"])).assign(fold=k, role=role)   # 'split' would be misleading here
        write_frame(out, f"sharded/cv_folds/fold_{k}_{role}")

for root, _, files in os.walk(LOCAL_DIR):
    for f in sorted(files):
        p = os.path.join(root, f)
        print(f"{p:55s} {len(read_frame(p)):5d} rows")

gcs_upload/sharded/dev.parquet                             16 rows
gcs_upload/sharded/new_data.parquet                        12 rows
gcs_upload/sharded/test.parquet                            16 rows
gcs_upload/sharded/train.parquet                           71 rows
gcs_upload/sharded/cv_folds/fold_0_train.parquet           29 rows
gcs_upload/sharded/cv_folds/fold_0_val.parquet              8 rows
gcs_upload/sharded/cv_folds/fold_1_train.parquet           37 rows
gcs_upload/sharded/cv_folds/fold_1_val.parquet              8 rows
gcs_upload/sharded/cv_folds/fold_2_train.parquet           45 rows
gcs_upload/sharded/cv_folds/fold_2_val.parquet              8 rows
gcs_upload/sharded/cv_folds/fold_3_train.parquet           53 rows
gcs_upload/sharded/cv_folds/fold_3_val.parquet              8 rows
gcs_upload/sharded/cv_folds/fold_4_train.parquet           61 rows
gcs_upload/sharded/cv_folds/fold_4_val.parquet              8 rows


In [ ]:
# CELL 18 - Validate the exported files BEFORE uploading (round trip, ordering, no leakage)
S = f"{LOCAL_DIR}/sharded"
ext = EXPORT_FORMAT
files = {n: read_frame(f"{S}/{n}.{ext}") for n in ["train", "dev", "test", "new_data"]}

# 1. round trip: what was read back equals what is in memory
for n, d in files.items():
    assert len(d) == len(export_frames[n]), f"{n}: row count changed on disk"

# 2. target quarters are strictly ordered across splits (no overlap = no leakage)
order = ["train", "dev", "test", "new_data"]
for a, b in zip(order, order[1:]):
    a_max = pd.PeriodIndex(files[a]["target_quarter"], freq="Q").max()
    b_min = pd.PeriodIndex(files[b]["target_quarter"], freq="Q").min()
    assert a_max < b_min, f"{a} / {b} target quarters overlap"

# 3. the test window is exactly what the mentor asked for
t = pd.PeriodIndex(files["test"]["target_quarter"], freq="Q")
assert t.min() == DEV_END + 1 and t.max() == TEST_END
assert pd.PeriodIndex(files["new_data"]["target_quarter"], freq="Q").min() == NEW_DATA_START

# 4. CV folds: chronological, inside the train split, never touching dev / test / new_data
train_last = files["train"]["reporting_date"].max()
for k in cv_folds:
    ft = read_frame(f"{S}/cv_folds/fold_{k}_train.{ext}")
    fv = read_frame(f"{S}/cv_folds/fold_{k}_val.{ext}")
    assert ft["reporting_date"].max() < fv["reporting_date"].min(), f"fold {k} leaks"
    assert fv["reporting_date"].max() <= train_last, f"fold {k} val outside the train split"

print("All export checks passed: files round-trip, splits are chronological, folds are leak-free")

All export checks passed: files round-trip, splits are chronological, folds are leak-free


In [ ]:
# CELL 19 - Upload the sharded files to gs://<bucket>/sharded/ (same layout as the local folder)
for root, _, files_ in os.walk(f"{LOCAL_DIR}/sharded"):
    for f in sorted(files_):
        local = os.path.join(root, f)
        upload_to_gcs(BUCKET_NAME, local, os.path.relpath(local, LOCAL_DIR).replace(os.sep, "/"))

Uploaded gcs_upload/sharded/dev.parquet -> gs://mortgage_irish_forecast_trends/sharded/dev.parquet
Uploaded gcs_upload/sharded/new_data.parquet -> gs://mortgage_irish_forecast_trends/sharded/new_data.parquet
Uploaded gcs_upload/sharded/test.parquet -> gs://mortgage_irish_forecast_trends/sharded/test.parquet
Uploaded gcs_upload/sharded/train.parquet -> gs://mortgage_irish_forecast_trends/sharded/train.parquet
Uploaded gcs_upload/sharded/cv_folds/fold_0_train.parquet -> gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_0_train.parquet
Uploaded gcs_upload/sharded/cv_folds/fold_0_val.parquet -> gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_0_val.parquet
Uploaded gcs_upload/sharded/cv_folds/fold_1_train.parquet -> gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_1_train.parquet
Uploaded gcs_upload/sharded/cv_folds/fold_1_val.parquet -> gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_1_val.parquet
Uploaded gcs_upload/sharded/cv_folds/fold_2_train.parque

## Section F - Queryable store in BigQuery (CELLS 20-21)

Mentor: *"you'll need a solution that allows quick and regular queries"*. GCS files are good for archiving and model training, but repeated EDA needs a SQL engine. BigQuery holds three kinds of table:

| Object | Purpose |
|---|---|
| `moa_tidy_long` | the cleaned long table (every category, row and unit) - explore any measure in one query |
| `features_<version>` | the modelling feature table with the `split` column - one source of truth for splits |
| `coop_features_<version>` | co-operation / restructure features (2019+) for goal 3 |
| views `features_<version>_train / _dev / _test / _new_data` | ready-made split tables |

In [ ]:
# CELL 20 - Load the tidy table, feature tables and split views into BigQuery
bq = bigquery.Client(project=PROJECT_ID)
dataset = bigquery.Dataset(f"{PROJECT_ID}.{BQ_DATASET}")
dataset.location = LOCATION
bq.create_dataset(dataset, exists_ok=True)

def load_df_to_bq(d, table):
    """Load a snake_case DataFrame, replacing the table. reporting_date is stored as a DATE."""
    out = d.copy()
    schema = []
    if "reporting_date" in out.columns:
        out["reporting_date"] = out["reporting_date"].dt.date
        schema.append(bigquery.SchemaField("reporting_date", "DATE"))
    cfg = bigquery.LoadJobConfig(write_disposition="WRITE_TRUNCATE", schema=schema)
    bq.load_table_from_dataframe(out, f"{PROJECT_ID}.{BQ_DATASET}.{table}", job_config=cfg).result()
    print(f"loaded {BQ_DATASET}.{table}: {out.shape}")

tidy_cols = ["Reporting Date", "Entity Group", "Format", "Category", "Row Position",
             "Column Position", "Row", "Observation Type", "unit", "value"]
tidy = to_snake_frame(df[tidy_cols]).assign(data_vintage=DATA_PULL_DATE)

load_df_to_bq(tidy, TBL_TIDY)
load_df_to_bq(to_snake_frame(feat), TBL_FEAT)
load_df_to_bq(to_snake_frame(coop_feat.reset_index()).assign(data_vintage=DATA_PULL_DATE), TBL_COOP)

# one view per split, so downstream code never re-implements the split logic
for s in ["train", "dev", "test", "new_data"]:
    bq.query(f"""
        CREATE OR REPLACE VIEW `{PROJECT_ID}.{BQ_DATASET}.{TBL_FEAT}_{s}` AS
        SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{TBL_FEAT}` WHERE split = '{s}'
    """).result()
print("views created:", [f"{TBL_FEAT}_{s}" for s in ["train", "dev", "test", "new_data"]])

loaded mortgage_arrears.moa_tidy_long: (19199, 11)
loaded mortgage_arrears.features_v1: (115, 15)
loaded mortgage_arrears.coop_features_v1: (81, 9)
views created: ['features_v1_train', 'features_v1_dev', 'features_v1_test', 'features_v1_new_data']


In [ ]:
# CELL 21 - Example EDA queries (these run in seconds and can be re-run any time)
def q(sql):
    return bq.query(sql).to_dataframe()

FT = f"`{PROJECT_ID}.{BQ_DATASET}.{TBL_FEAT}`"
TD = f"`{PROJECT_ID}.{BQ_DATASET}.{TBL_TIDY}`"

print("1) Split sizes and coverage")
display(q(f"""
    SELECT split, COUNT(*) AS n_rows, COUNT(DISTINCT reporting_date) AS n_quarters,
           MIN(reporting_date) AS first_feature_q, MAX(reporting_date) AS last_feature_q,
           MIN(target_quarter) AS first_target_q, MAX(target_quarter) AS last_target_q
    FROM {FT} GROUP BY split ORDER BY first_feature_q
"""))

print("2) Yearly average arrears and long-arrears share by loan format")
display(q(f"""
    SELECT EXTRACT(YEAR FROM reporting_date) AS year, loan_format,
           ROUND(AVG(arrears_total)) AS avg_arrears_accounts,
           ROUND(AVG(over180_share), 3) AS avg_over180_share
    FROM {FT} GROUP BY year, loan_format ORDER BY year, loan_format
"""))

print("3) Which categories and measures exist in the raw data, and since when?")
display(q(f"""
    SELECT category, unit, COUNT(DISTINCT row_label) AS n_measures,
           MIN(reporting_date) AS first_q, MAX(reporting_date) AS last_q
    FROM {TD} GROUP BY category, unit ORDER BY category, unit
"""))

1) Split sizes and coverage


,split,n_rows,n_quarters,first_feature_q,last_feature_q,first_target_q,last_target_q
0,train,71,41,2010-09-30,2020-09-30,2010Q4,2020Q4
1,dev,16,8,2020-12-31,2022-09-30,2021Q1,2022Q4
2,test,16,8,2022-12-30,2024-09-30,2023Q1,2024Q4
3,new_data,12,6,2024-12-31,2026-03-31,2025Q1,2026Q2


2) Yearly average arrears and long-arrears share by loan format


,year,loan_format,avg_arrears_accounts,avg_over180_share
0,2010,PDH,87798.0,0.338
1,2011,PDH,106494.0,0.404
2,2012,PDH,132938.0,0.496
3,2013,BTL,39865.0,0.657
4,2013,PDH,140709.0,0.570
5,2014,BTL,38269.0,0.720
6,2014,PDH,121619.0,0.620
7,2015,BTL,30993.0,0.755
8,2015,PDH,95875.0,0.635
9,2016,BTL,27279.0,0.755


3) Which categories and measures exist in the raw data, and since when?


,category,unit,n_measures,first_q,last_q
0,Arrears,Arrears,10,2009-09-30,2026-03-31
1,Arrears,Balance,10,2009-09-30,2026-03-31
2,Arrears,Number,10,2009-09-30,2026-03-31
3,Borrower co-operation status,"Not Co-operating, Legal Proceed. Initiate",7,2019-09-30,2026-03-31
4,Borrower co-operation status,"Not Co-operating, Legal Proceed. Not Initiate",7,2019-09-30,2026-03-31
5,Borrower co-operation status,"Number, In Arrears, Restruc.",7,2019-09-30,2026-03-31
6,Borrower co-operation status,"Number, Not Restruc., Co-operating",7,2019-09-30,2026-03-31
7,Duration in Legal Proceedings,Arrears,4,2019-09-30,2026-03-31
8,Duration in Legal Proceedings,Balance,4,2019-09-30,2026-03-31
9,Duration in Legal Proceedings,Number,4,2019-09-30,2026-03-31


## Section G - Documentation artefacts (CELLS 22-25)

In [ ]:
# CELL 22 - Build metadata.json from the live notebook state (numbers cannot drift from what ran)
def split_info(d):
    return {"rows": int(len(d)),
            "feature_quarters": int(d["reporting_date"].nunique()),
            "first_feature_quarter": str(d["reporting_date"].min().date()),
            "last_feature_quarter": str(d["reporting_date"].max().date()),
            "first_target_quarter": str(d["target_quarter"].min()),
            "last_target_quarter": str(d["target_quarter"].max()),
            "rows_without_target": int(d["target_next_q"].isna().sum())}

FEATURE_DESCRIPTIONS = {
    "reporting_date":  "Quarter-end date of the FEATURES (dd/mm/yyyy in source, parsed to a date)",
    "entity_group":    "Lender group: 'All' = Banks + Non-Banks combined; otherwise 'Banks' or 'Non-Banks'",
    "loan_format":     "PDH = principal dwelling home; BTL = buy-to-let",
    "arrears_total":   "Number of mortgage accounts in arrears (all buckets)",
    "over180_share":   "Accounts in arrears over 180 days / total accounts in arrears (rebuilt as a residual)",
    "b91_180_growth":  "Quarter-on-quarter % change in accounts 91-180 days in arrears (inflow signal)",
    "arrears_rate":    "Accounts in arrears / total mortgage accounts (only when STOCK_ROW is set)",
    "target_next_q":   f"Next quarter's value of '{TARGET}' (forecast label); empty for the newest quarter",
    "target_quarter":  "Quarter the label belongs to (feature quarter + 1); the split is defined on this column",
    "split":           "train / dev / test / new_data (historical, by target quarter)",
    "data_vintage":    "Date the raw file was pulled (publisher revises past figures)",
    "feature_version": "Feature-set version that produced the row",
    "fold": "CV fold number (cv_folds files only)",
    "role": "'train' or 'val' inside a CV fold (cv_folds files only)",
}
for lag in N_LAGS:
    FEATURE_DESCRIPTIONS[f"{TARGET}_lag{lag}"] = f"'{TARGET}' lagged {lag} quarter(s)"

cols_present = list(export_frames["train"].columns)

metadata = {
    "dataset": "Residential Mortgage Arrears and Repossession Statistics",
    "publisher": "Central Bank of Ireland",
    "source_url": "https://data.gov.ie/dataset/residential-mortgage-arrears-and-repossession-statistics",
    "licence": "CC BY 4.0",
    "source_coverage": "2009-09-30 to latest release, subject to revision",
    "raw_file": {
        "gcs_path": "raw/moa-open-data.csv",
        "encoding": "cp1252",
        "sha256": RAW_SHA256,
        "size_bytes": os.path.getsize(LOCAL_RAW_PATH),
        "rows": int(len(df)),
        "pulled_on": DATA_PULL_DATE,
    },
    "storage": {
        "raw_and_exports": f"gs://{BUCKET_NAME} (raw/, sharded/), object versioning on",
        "queryable_store": f"BigQuery {PROJECT_ID}.{BQ_DATASET}",
        "tables": [TBL_TIDY, TBL_FEAT, TBL_COOP],
        "split_views": [f"{TBL_FEAT}_{s}" for s in ["train", "dev", "test", "new_data"]],
        "export_format": EXPORT_FORMAT,
    },
    "processing": {
        "entity_used": ENTITY,
        "target": TARGET,
        "stock_row": STOCK_ROW,
        "over_180_method": OVER_180_METHOD,
        "over_180_overlap_max_abs_diff": OVER_180_OVERLAP_MAX_DIFF,
        "value_units": "account counts as reported; EUR (000) converted to EUR where money columns are used",
        "lags": list(N_LAGS),
        "feature_version": FEATURE_VERSION,
        "co_operation_features": f"in {TBL_COOP} (data from 2019-09-30); not part of the split files",
    },
    "split": {
        "strategy": "historical, defined on the TARGET quarter; whole quarters go to one partition",
        "boundaries": {"train_target_end": str(TRAIN_END), "dev_target_end": str(DEV_END),
                       "test_target_end": str(TEST_END), "new_data_target_start": str(NEW_DATA_START)},
        "train": split_info(export_frames["train"]),
        "dev": split_info(export_frames["dev"]),
        "test": split_info(export_frames["test"]),
        "new_data": split_info(export_frames["new_data"]),
        "new_data_note": "held back for sprint 4; not used for training, tuning or testing",
    },
    "cross_validation": {
        "type": "expanding window on train only",
        "n_splits": len(cv_folds),
        "val_quarters_per_fold": CV_VAL_QUARTERS,
        "gap_quarters": CV_GAP,
        "folds": {str(k): {"train_last_quarter": str(tr["Reporting Date"].max().date()),
                           "val_first_quarter": str(va["Reporting Date"].min().date()),
                           "val_last_quarter": str(va["Reporting Date"].max().date())}
                  for k, (tr, va) in cv_folds.items()},
    },
    "columns": {c: {"dtype": str(export_frames["train"][c].dtype),
                    "description": FEATURE_DESCRIPTIONS.get(c, "TODO: describe")} for c in cols_present},
    "created_on": datetime.datetime.now().isoformat(timespec="seconds"),
}

print("Columns without a description:", [c for c in cols_present if c not in FEATURE_DESCRIPTIONS])
METADATA_PATH = f"{LOCAL_DIR}/sharded/metadata.json"
with open(METADATA_PATH, "w") as f:
    json.dump(metadata, f, indent=2)
print(json.dumps(metadata["split"], indent=2)[:1800])

Columns without a description: []
{
  "strategy": "historical, defined on the TARGET quarter; whole quarters go to one partition",
  "boundaries": {
    "train_target_end": "2020Q4",
    "dev_target_end": "2022Q4",
    "test_target_end": "2024Q4",
    "new_data_target_start": "2025Q1"
  },
  "train": {
    "rows": 71,
    "feature_quarters": 41,
    "first_feature_quarter": "2010-09-30",
    "last_feature_quarter": "2020-09-30",
    "first_target_quarter": "2010Q4",
    "last_target_quarter": "2020Q4",
    "rows_without_target": 0
  },
  "dev": {
    "rows": 16,
    "feature_quarters": 8,
    "first_feature_quarter": "2020-12-31",
    "last_feature_quarter": "2022-09-30",
    "first_target_quarter": "2021Q1",
    "last_target_quarter": "2022Q4",
    "rows_without_target": 0
  },
  "test": {
    "rows": 16,
    "feature_quarters": 8,
    "first_feature_quarter": "2022-12-30",
    "last_feature_quarter": "2024-09-30",
    "first_target_quarter": "2023Q1",
    "last_target_quarter": "2024

In [ ]:
# CELL 23 - Write a human-readable data card from the metadata
sp = metadata["split"]
def row(name, key):
    s = sp[key]
    return (f"| {name} | {s['feature_quarters']} | {s['first_feature_quarter']} to {s['last_feature_quarter']} "
            f"| {s['first_target_quarter']} to {s['last_target_quarter']} |")

card = f"""# Data Card: Irish Mortgage Arrears Forecasting Dataset

## Source
Central Bank of Ireland, *Residential Mortgage Arrears and Repossession Statistics*
({metadata['source_url']}). Licence: {metadata['licence']}.
Quarterly, {metadata['source_coverage']}. Pulled on {DATA_PULL_DATE}.
Raw file: `raw/moa-open-data.csv` (SHA-256 `{RAW_SHA256[:16]}...`).

## What one row is
One observation per (quarter, format) for the modelling entity **{ENTITY}**.
Features come from quarter t; the label `target_next_q` is **{TARGET}** at quarter t+1.

## Split (historical, defined on the target quarter)
| Split | Feature quarters | Feature date range | Target quarters |
|---|---|---|---|
{row('Train', 'train')}
{row('Dev', 'dev')}
{row('Test', 'test')}
{row('New data (sprint 4)', 'new_data')}

Why by target quarter: a row's label lives one quarter after its features, so splitting on the
feature date would let a later value leak into the labels of the previous split.
`new_data` is held back untouched. Its newest row has no label yet and is the row to predict from.
CV: expanding window on train only, {len(cv_folds)} folds, {CV_VAL_QUARTERS} validation quarters each, {CV_GAP}-quarter gap.

## Storage
GCS `gs://{BUCKET_NAME}`: `raw/` (untouched source) and `sharded/` ({EXPORT_FORMAT} exports, object versioning on).
BigQuery `{PROJECT_ID}.{BQ_DATASET}`: `{TBL_TIDY}`, `{TBL_FEAT}`, `{TBL_COOP}` and one view per split.

## Known issues and decisions
- Values were text in places; coerced to numeric (thousands separators removed).
- The "over 180 days" row exists only for 2009 to mid-2012. Rebuilt as: {OVER_180_METHOD}
  (overlap check, max abs diff: {OVER_180_OVERLAP_MAX_DIFF}).
- Banks and Non-Banks are combined for the national series, so loan-book sales between them cancel out.
- BTL data starts 2012; PDH starts 2009.
- Co-operation / restructure features exist only from 2019-09-30 and are kept in `{TBL_COOP}`;
  too few train quarters overlap them to train a model on this split.
- The publisher revises past figures; `data_vintage` records the pull date of every row.
- Small sample and strong distribution shift (arrears fell sharply after 2013): dev and test are calmer
  than the training crisis period, so always report naive baselines next to model scores.

## Columns
| Column | dtype | Description |
|---|---|---|
""" + "\n".join(f"| {c} | {v['dtype']} | {v['description']} |" for c, v in metadata["columns"].items()) + "\n"

CARD_PATH = f"{LOCAL_DIR}/sharded/DATA_CARD.md"
with open(CARD_PATH, "w") as f:
    f.write(card)
print(card[:2200])

# Data Card: Irish Mortgage Arrears Forecasting Dataset

## Source
Central Bank of Ireland, *Residential Mortgage Arrears and Repossession Statistics*
(https://data.gov.ie/dataset/residential-mortgage-arrears-and-repossession-statistics). Licence: CC BY 4.0.
Quarterly, 2009-09-30 to latest release, subject to revision. Pulled on 2026-09-30.
Raw file: `raw/moa-open-data.csv` (SHA-256 `67b8ac6b677928f8...`).

## What one row is
One observation per (quarter, format) for the modelling entity **All**.
Features come from quarter t; the label `target_next_q` is **arrears_total** at quarter t+1.

## Split (historical, defined on the target quarter)
| Split | Feature quarters | Feature date range | Target quarters |
|---|---|---|---|
| Train | 41 | 2010-09-30 to 2020-09-30 | 2010Q4 to 2020Q4 |
| Dev | 8 | 2020-12-31 to 2022-09-30 | 2021Q1 to 2022Q4 |
| Test | 8 | 2022-12-30 to 2024-09-30 | 2023Q1 to 2024Q4 |
| New data (sprint 4) | 6 | 2024-12-31 to 2026-03-31 | 2025Q1 to 2026Q2 |

Why by targe

In [ ]:
# CELL 24 - Upload metadata.json and DATA_CARD.md next to the data
upload_to_gcs(BUCKET_NAME, METADATA_PATH, "sharded/metadata.json")
upload_to_gcs(BUCKET_NAME, CARD_PATH, "sharded/DATA_CARD.md")

# list the bucket so you can see the final layout
for b in storage.Client(project=PROJECT_ID).list_blobs(BUCKET_NAME):
    print(f"gs://{BUCKET_NAME}/{b.name}")

Uploaded gcs_upload/sharded/metadata.json -> gs://mortgage_irish_forecast_trends/sharded/metadata.json
Uploaded gcs_upload/sharded/DATA_CARD.md -> gs://mortgage_irish_forecast_trends/sharded/DATA_CARD.md
gs://mortgage_irish_forecast_trends/processed/mortgage_arrears_cv_folds.csv
gs://mortgage_irish_forecast_trends/processed/mortgage_arrears_splits.csv
gs://mortgage_irish_forecast_trends/raw/
gs://mortgage_irish_forecast_trends/raw/moa-open-data.csv
gs://mortgage_irish_forecast_trends/sharded/DATA_CARD.md
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_0_train.csv
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_0_train.parquet
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_0_val.csv
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_0_val.parquet
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_1_train.csv
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_1_train.parquet
gs://mortgage_irish_forecast_trends/sharded/cv_folds/fold_1_va

In [ ]:
# CELL 25 - How to load the data back later (for the modelling notebooks)

# (a) from BigQuery: fastest for exploration, the split logic already lives in the views
train_bq = bq.query(f"SELECT * FROM `{PROJECT_ID}.{BQ_DATASET}.{TBL_FEAT}_train`").to_dataframe()

# (b) from the Parquet files in GCS: good for training jobs (needs gcsfs; pandas reads gs:// paths directly)
base = f"gs://{BUCKET_NAME}/sharded"
train_gcs = pd.read_parquet(f"{base}/train.parquet")
test_gcs  = pd.read_parquet(f"{base}/test.parquet")
f0_tr = pd.read_parquet(f"{base}/cv_folds/fold_0_train.parquet")
f0_va = pd.read_parquet(f"{base}/cv_folds/fold_0_val.parquet")

# new_data (2025 onward) stays untouched until sprint 4:
# new_gcs = pd.read_parquet(f"{base}/new_data.parquet")
print(train_bq.shape, train_gcs.shape, test_gcs.shape, f0_tr.shape, f0_va.shape)

(71, 15) (71, 15) (16, 15) (29, 16) (8, 16)
